In [0]:
# PURPOSE:
# This notebook creates the central Fact table of our
# E-Commerce Sales Analytics Star Schema.
#
# STAR SCHEMA DESIGN:
#
#                    dim_customer
#                         |
#                         |
# dim_product ----> fact_sales <---- dim_date
#
# fact_sales is the central table because it contains the
# measurable business events and metrics related to each order.
#
# DIMENSION TABLES:
# 1. dim_customer -> WHO placed the order
# 2. dim_product  -> WHAT product was ordered
# 3. dim_date     -> WHEN the order happened
#
# FACT TABLE:
# fact_sales contains:
# - Order-level transaction information
# - Foreign/reference keys to the dimension tables
# - Quantity and sales/revenue measures
# - Order status and payment information
#
# KEY RELATIONSHIPS:
#
# dim_customer.customer_key  -> fact_sales.customer_key
# dim_product.product_key    -> fact_sales.product_key
# dim_date.date_key          -> fact_sales.date_key
#
# Here, the dimension keys are used in the fact table to connect
# each sales transaction with its customer, product, and date.
#
# WHY THIS NOTEBOOK IS NEEDED:
# The fact table provides a single central place for analyzing
# sales transactions. The Gold analytics notebook can then use
# fact_sales together with the dimensions to calculate:
# - Total Sales
# - Monthly Revenue
# - Revenue Trends
# - State/City Sales
# - Top Customers
# - Best Products
# - Category and Brand Performance
# - Payment and Coupon Performance
#
# DATA FLOW:
#
# Silver Tables
#      ↓
# Gold Dimensions + Gold Fact
#      ↓
# Star Schema
#      ↓
# Gold Analytics
#
# IMPORTANT:
# This notebook focuses on building the business-ready Fact table.

In [0]:
sales = spark.table(
    "workspace.indian_ecommerce_sales_analytics.silver_sales"
)

dim_customer = spark.table(
    "workspace.indian_ecommerce_sales_analytics.dim_customer"
)

dim_product = spark.table(
    "workspace.indian_ecommerce_sales_analytics.dim_product"
)

dim_date = spark.table(
    "workspace.indian_ecommerce_sales_analytics.dim_date"
)

print("Sales:", sales.count())
print("Customers:", dim_customer.count())
print("Products:", dim_product.count())
print("Dates:", dim_date.count())

Sales: 250000
Customers: 40000
Products: 2000
Dates: 760


In [0]:
#creating date key
from pyspark.sql import functions as F
fact_sales = (
    sales
    .withColumn(
        "Date_Key",
        F.date_format(
            F.col("Order_Date"),
            "yyyyMMdd"
        ).cast("int")
    )
)

display(
    fact_sales.select(
        "Order_ID",
        "Order_Date",
        "Date_Key"
    ).limit(10)
)

Order_ID,Order_Date,Date_Key
ORD0000000001,2026-06-07,20260607
ORD0000000002,2025-02-04,20250204
ORD0000000003,2026-03-12,20260312
ORD0000000004,2025-01-27,20250127
ORD0000000005,2024-06-30,20240630
ORD0000000006,2025-09-18,20250918
ORD0000000007,2026-03-01,20260301
ORD0000000008,2024-06-19,20240619
ORD0000000009,2025-02-12,20250212
ORD0000000010,2024-10-19,20241019


In [0]:
#Validating date keys
missing_dates = (
    fact_sales
    .select("Date_Key")
    .distinct()
    .join(
        dim_date.select("Date_Key").distinct(),
        on="Date_Key",
        how="left_anti"
    )
)

print(
    "Sales records with missing Date Keys:",
    missing_dates.count()
)

Sales records with missing Date Keys: 0


In [0]:
#selecting the transaction columns
fact_sales = fact_sales.select(
    "Order_ID",
    "Customer_ID",
    "Product_ID",
    "Date_Key",
    "Order_Date",
    "Order_Time",
    "Delivery_Date",
    "Delivery_Days",
    "Quantity",
    "Unit_Price",
    "Order_Value",
    "Shipping_Cost",
    "Coupon_Code",
    "Coupon_Discount",
    "Total_Amount",
    "Calculated_Total_Amount",
    "Amount_Difference",
    "Payment_Mode",
    "Order_Status",
    "Rating",
    "Review_Text",
    "City",
    "State",
    "Customer_Age",
    "Customer_Age_Group",
    "Year",
    "Month",
    "Month_Name",
    "Quarter",
    "Year_Month",
    "Day",
    "Day_of_Week",
    "Has_Coupon",
    "City_Tier",
    "dq_negative_total_amount",
    "dq_amount_mismatch",
    "dq_invalid_quantity",
    "dq_invalid_unit_price",
    "dq_invalid_order_value",
    "dq_invalid_coupon_discount",
    "dq_invalid_rating",
    "dq_invalid_delivery_date"
)

In [0]:
total_rows = fact_sales.count()

distinct_orders = (
    fact_sales
    .select("Order_ID")
    .distinct()
    .count()
)

print("Total Fact Rows:", total_rows)
print("Distinct Orders:", distinct_orders)

Total Fact Rows: 250000
Distinct Orders: 250000


In [0]:
#customer table foreign key validation
missing_customer_fk = (
    fact_sales
    .select("Customer_ID")
    .distinct()
    .join(
        dim_customer
        .select("Customer_ID")
        .distinct(),
        on="Customer_ID",
        how="left_anti"
    )
)

print(
    "Missing Customer Foreign Keys:",
    missing_customer_fk.count()
)

Missing Customer Foreign Keys: 0


In [0]:
#product table foreign key validation
missing_product_fk = (
    fact_sales
    .select("Product_ID")
    .distinct()
    .join(
        dim_product
        .select("Product_ID")
        .distinct(),
        on="Product_ID",
        how="left_anti"
    )
)

print(
    "Missing Product Foreign Keys:",
    missing_product_fk.count()
)
#Date foreign key validation
missing_date_fk = (
    fact_sales
    .select("Date_Key")
    .distinct()
    .join(
        dim_date
        .select("Date_Key")
        .distinct(),
        on="Date_Key",
        how="left_anti"
    )
)

print(
    "Missing Date Foreign Keys:",
    missing_date_fk.count()
)

Missing Product Foreign Keys: 0
Missing Date Foreign Keys: 0


In [0]:
F.col("Order_Status") == "Delivered"

Column<'==(Order_Status, Delivered)'>

In [0]:
#realized revenue flag
fact_sales = fact_sales.withColumn(
    "Is_Delivered",
    F.when(
        F.col("Order_Status") == "Delivered",
        True
    ).otherwise(False)
)
display(
    fact_sales
    .groupBy("Order_Status", "Is_Delivered")
    .count()
    .orderBy(F.desc("count"))
)

Order_Status,Is_Delivered,count
Delivered,true,200139
Cancelled,false,12507
Returned,false,12493
Shipped,false,12459
Processing,false,12402


In [0]:
#Realized revenue amount
fact_sales = fact_sales.withColumn(
    "Realized_Revenue",
    F.when(
        F.col("Is_Delivered"),
        F.col("Total_Amount")
    ).otherwise(F.lit(0.0))
)

In [0]:
display(
    fact_sales.select(
        "Order_ID",
        "Customer_ID",
        "Product_ID",
        "Date_Key",
        "Order_Status",
        "Total_Amount",
        "Is_Delivered",
        "Realized_Revenue"
    ).limit(20)
)

Order_ID,Customer_ID,Product_ID,Date_Key,Order_Status,Total_Amount,Is_Delivered,Realized_Revenue
ORD0000000001,CUST00014303,PROD001017,20260607,Delivered,2622.46,true,2622.46
ORD0000000002,CUST00034253,PROD000703,20250204,Delivered,90258.0,true,90258.0
ORD0000000003,CUST00000421,PROD000318,20260312,Processing,3536.5,false,0.0
ORD0000000004,CUST00026507,PROD000213,20250127,Delivered,34031.56,true,34031.56
ORD0000000005,CUST00009418,PROD001437,20240630,Delivered,138042.34,true,138042.34
ORD0000000006,CUST00033802,PROD000121,20250918,Returned,4439.77,false,0.0
ORD0000000007,CUST00027469,PROD001308,20260301,Processing,67241.98,false,0.0
ORD0000000008,CUST00017326,PROD001392,20240619,Delivered,2090.92,true,2090.92
ORD0000000009,CUST00009627,PROD001876,20250212,Delivered,20211.02,true,20211.02
ORD0000000010,CUST00012719,PROD000214,20241019,Delivered,86647.71,true,86647.71


In [0]:
fact_sales.printSchema()

root
 |-- Order_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Date_Key: integer (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Order_Time: timestamp (nullable = true)
 |-- Delivery_Date: date (nullable = true)
 |-- Delivery_Days: integer (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Unit_Price: double (nullable = true)
 |-- Order_Value: double (nullable = true)
 |-- Shipping_Cost: double (nullable = true)
 |-- Coupon_Code: string (nullable = true)
 |-- Coupon_Discount: double (nullable = true)
 |-- Total_Amount: double (nullable = true)
 |-- Calculated_Total_Amount: double (nullable = true)
 |-- Amount_Difference: double (nullable = true)
 |-- Payment_Mode: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Review_Text: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |--

In [0]:
print("Fact Rows:", fact_sales.count())

print(
    "Distinct Order IDs:",
    fact_sales
    .select("Order_ID")
    .distinct()
    .count()
)

print(
    "Distinct Customers:",
    fact_sales
    .select("Customer_ID")
    .distinct()
    .count()
)

print(
    "Distinct Products:",
    fact_sales
    .select("Product_ID")
    .distinct()
    .count()
)

print(
    "Distinct Dates:",
    fact_sales
    .select("Date_Key")
    .distinct()
    .count()
)

Fact Rows: 250000
Distinct Order IDs: 250000
Distinct Customers: 39914
Distinct Products: 2000
Distinct Dates: 760


In [0]:
(
    fact_sales.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.indian_ecommerce_sales_analytics.fact_sales"
    )
)

In [0]:
%sql
SELECT COUNT(*) AS fact_sales_records
FROM workspace.indian_ecommerce_sales_analytics.fact_sales;

fact_sales_records
250000


In [0]:
%sql
SELECT
    COUNT(DISTINCT Order_ID) AS unique_orders,
    COUNT(DISTINCT Customer_ID) AS unique_customers,
    COUNT(DISTINCT Product_ID) AS unique_products,
    COUNT(DISTINCT Date_Key) AS unique_dates
FROM workspace.indian_ecommerce_sales_analytics.fact_sales;

unique_orders,unique_customers,unique_products,unique_dates
250000,39914,2000,760
